# Proyek P08 — Eksperimen & Pemodelan Machine Learning (ML Engineer)
**Prediksi Performa Akademik Mahasiswa (Pendekatan Regresi Nilai Akhir)**

Notebook ini merupakan artefak utama **ML Engineer** sesuai **Bab 4 Panduan Pengerjaan Proyek P08**. Seluruh alur dirancang untuk mencegah kebocoran informasi (*data/target leakage*), membandingkan model secara adil dengan protokol *5-Fold Cross-Validation*, mendiagnosis *overfitting* melalui *learning curve*, menjelaskan faktor penentu prediksi dengan **SHAP (`TreeExplainer`)**, serta menghasilkan paket model siap pakai untuk **App Developer** melalui [predictor.py](file:///c:/Users/MARC/predictionmachine/predictor.py).

### Alur Pengerjaan
1. **Persiapan Lingkungan & Fungsi Evaluasi Standar**: Mengatur `RANDOM_STATE = 42`, fungsi `evaluate()` (RMSE, MAE, $R^2$ pada Train dan Test), serta fungsi plot diagnostik (*Learning Curve*, Prediksi vs Aktual, dan Residual).
2. **Inspeksi Data & Pencegahan *Target Leakage***: Memuat data bersih unscaled, menetapkan target regresi `curricular_units_2nd_sem_grade` (skala `0.0 - 20.0`), dan membuang atribut Semester 2 lainnya serta status akhir `target` agar tidak membocorkan masa depan.
3. **Train-Test Split & `ColumnTransformer` di Dalam Pipeline**: Membagi data 80% Train / 20% Test (`random_state=42`) sebelum transformasi apa pun, lalu membungkus `OneHotEncoder(handle_unknown="ignore")` di dalam `Pipeline` tanpa normalisasi (`StandardScaler`).
4. **Pemodelan & Hyperparameter Tuning (5-Fold CV)**:
   - **Baseline**: `DummyRegressor(strategy="mean")`
   - **Model 1**: `DecisionTreeRegressor` (Default vs Tuned: `max_depth`, `min_samples_leaf`, `min_samples_split`, `ccp_alpha`)
   - **Model 2**: `RandomForestRegressor` (Default vs Tuned: `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features`)
5. **Evaluasi Akhir, *Learning Curve*, & Analisis Residual/Subpopulasi**: Menggunakan data Test **hanya satu kali di akhir** disertai interval kepercayaan 95% (*bootstrapping*).
6. **Interpretabilitas SHAP (`TreeExplainer`) & Serah Terima**: Agregasi SHAP per fitur asli (menggabungkan pecahan *One-Hot Encoding*), visualisasi global & lokal, penyimpanan `model/final_pipeline.joblib`, dan verifikasi `predict_and_explain()`.

In [ ]:
# ==============================================================================
# 1. SETUP LINGKUNGAN, IMPORT LIBRARY & REPRODUSIBILITAS
# ==============================================================================
import json
import os
import re
import warnings
from typing import Any, Dict, List, Tuple

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import (
    GridSearchCV,
    KFold,
    learning_curve,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeRegressor

try:
    import shap
except ImportError:
    raise ImportError(
        "Library 'shap' belum terpasang di kernel ini. Jalankan `%pip install shap matplotlib seaborn` di sel terpisah."
    )

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3
sns.set_theme(style="whitegrid", palette="deep")

print("Versi Library yang Digunakan (Kontrak Lingkungan Proyek P08):")
print(f"  - numpy        : {np.__version__}")
print(f"  - pandas       : {pd.__version__}")
print(f"  - scikit-learn : {sklearn.__version__}")
print(f"  - joblib       : {joblib.__version__}")
print(f"  - shap         : {shap.__version__}")
print(f"  - RANDOM_STATE : {RANDOM_STATE}")

### Analisis Setup Lingkungan & Reprodusibilitas
Sel di atas mengunci `RANDOM_STATE = 42` untuk `numpy` dan seluruh algoritma berbasis pohon (`DecisionTreeRegressor`, `RandomForestRegressor`, serta pembagian `KFold` dan `train_test_split`). Sesuai kontrak serah terima di **Bab 2 Panduan P08**, pencatatan versi `scikit-learn`, `joblib`, `pandas`, `numpy`, dan `shap` memastikan bahwa objek `Pipeline` yang disimpan ke `.joblib` dapat dimuat oleh **App Developer** tanpa galat ketidakcocokan serialisasi (*pickle/joblib version mismatch*).

---

## 2. Fungsi Evaluasi Standar & Visualisasi Diagnostik (Bab 4 Panduan P08)
Sesuai instruksi *"Selagi menunggu data bersih"* pada Bab 4, kita menyiapkan:
1. Fungsi `evaluate()` yang menghitung **RMSE** (`root_mean_squared_error`), **MAE** (`mean_absolute_error`), dan **$R^2$** (`r2_score`) pada data **Train** maupun **Test** dalam satu tabel perbandingan.
2. Fungsi `plot_learning_curve_rmse()` untuk mendeteksi gejala *overfitting* (selisih besar antara RMSE Train dan RMSE Cross-Validation).
3. Fungsi `plot_actual_vs_predicted()` dan `plot_residuals()` untuk memeriksa bias prediksi serta heteroskedastisitas error.

In [ ]:
# ==============================================================================
# 2. FUNGSI EVALUASI (RMSE, MAE, R²) & PLOT DIAGNOSTIK
# ==============================================================================
def evaluate(
    model: Any,
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    model_name: str = "Model",
) -> pd.DataFrame:
    """
    Menghitung RMSE, MAE, dan R² pada data Train dan Test dalam satu baris DataFrame.
    Sesuai Bab 4 Panduan P08, RMSE dihitung menggunakan `root_mean_squared_error`.
    """
    y_pred_train = model.predict(X_train)
    y_pred_test = model.predict(X_test)

    metrics = {
        "Model": model_name,
        "Train_RMSE": root_mean_squared_error(y_train, y_pred_train),
        "Test_RMSE": root_mean_squared_error(y_test, y_pred_test),
        "Gap_RMSE (Test-Train)": root_mean_squared_error(y_test, y_pred_test)
        - root_mean_squared_error(y_train, y_pred_train),
        "Train_MAE": mean_absolute_error(y_train, y_pred_train),
        "Test_MAE": mean_absolute_error(y_test, y_pred_test),
        "Train_R2": r2_score(y_train, y_pred_train),
        "Test_R2": r2_score(y_test, y_pred_test),
    }
    return pd.DataFrame([metrics])


def bootstrap_ci_metrics(
    y_true: pd.Series,
    y_pred: np.ndarray,
    n_bootstraps: int = 1000,
    random_state: int = RANDOM_STATE,
) -> Dict[str, Tuple[float, float]]:
    """Menghitung 95% Confidence Interval (CI) untuk RMSE, MAE, dan R² dengan bootstrapping."""
    rng = np.random.RandomState(random_state)
    y_true_arr = np.asarray(y_true)
    y_pred_arr = np.asarray(y_pred)
    n = len(y_true_arr)

    rmse_list, mae_list, r2_list = [], [], []
    for _ in range(n_bootstraps):
        idx = rng.randint(0, n, n)
        rmse_list.append(root_mean_squared_error(y_true_arr[idx], y_pred_arr[idx]))
        mae_list.append(mean_absolute_error(y_true_arr[idx], y_pred_arr[idx]))
        r2_list.append(r2_score(y_true_arr[idx], y_pred_arr[idx]))

    return {
        "RMSE_95CI": (float(np.percentile(rmse_list, 2.5)), float(np.percentile(rmse_list, 97.5))),
        "MAE_95CI": (float(np.percentile(mae_list, 2.5)), float(np.percentile(mae_list, 97.5))),
        "R2_95CI": (float(np.percentile(r2_list, 2.5)), float(np.percentile(r2_list, 97.5))),
    }


def plot_learning_curve_rmse(
    estimator: Any,
    X: pd.DataFrame,
    y: pd.Series,
    title: str,
    cv: Any,
    ax: plt.Axes,
) -> None:
    """Membuat plot Learning Curve berbasis RMSE pada data Train dan Cross-Validation."""
    train_sizes, train_scores, val_scores = learning_curve(
        estimator,
        X,
        y,
        cv=cv,
        scoring="neg_root_mean_squared_error",
        train_sizes=np.linspace(0.1, 1.0, 6),
        n_jobs=-1,
    )
    train_rmse_mean = -np.mean(train_scores, axis=1)
    train_rmse_std = np.std(-train_scores, axis=1)
    val_rmse_mean = -np.mean(val_scores, axis=1)
    val_rmse_std = np.std(-val_scores, axis=1)

    ax.plot(train_sizes, train_rmse_mean, "o-", color="#1f77b4", label="Train RMSE")
    ax.fill_between(
        train_sizes,
        train_rmse_mean - train_rmse_std,
        train_rmse_mean + train_rmse_std,
        alpha=0.15,
        color="#1f77b4",
    )
    ax.plot(train_sizes, val_rmse_mean, "s-", color="#d62728", label="CV (5-Fold) RMSE")
    ax.fill_between(
        train_sizes,
        val_rmse_mean - val_rmse_std,
        val_rmse_mean + val_rmse_std,
        alpha=0.15,
        color="#d62728",
    )
    ax.set_title(title, fontsize=11, fontweight="bold")
    ax.set_xlabel("Jumlah Sampel Latih (Train Size)")
    ax.set_ylabel("RMSE (Lebih Rendah = Lebih Baik)")
    ax.legend(loc="best")


def plot_actual_vs_predicted_and_residuals(
    model: Any,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    model_name: str,
) -> None:
    """Menampilkan berdampingan Plot Prediksi vs Aktual dan Plot Residual pada Data Test."""
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred

    fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

    # Subplot 1: Aktual vs Prediksi
    axes[0].scatter(y_test, y_pred, alpha=0.4, color="#2b5c8f", edgecolors="none", s=30)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    axes[0].plot(
        [min_val, max_val],
        [min_val, max_val],
        "r--",
        lw=2,
        label="Garis Ideal (Prediksi = Aktual)",
    )
    axes[0].set_title(f"{model_name} — Aktual vs Prediksi (Test Set)", fontweight="bold")
    axes[0].set_xlabel("Nilai Aktual Semester 2 (Skala 0-20)")
    axes[0].set_ylabel("Nilai Prediksi Semester 2 (Skala 0-20)")
    axes[0].legend()

    # Subplot 2: Plot Residual
    axes[1].scatter(y_pred, residuals, alpha=0.4, color="#d95f02", edgecolors="none", s=30)
    axes[1].axhline(0, color="black", linestyle="--", lw=2, label="Residual = 0")
    axes[1].set_title(f"{model_name} — Plot Residual (Aktual - Prediksi)", fontweight="bold")
    axes[1].set_xlabel("Nilai Prediksi Semester 2")
    axes[1].set_ylabel("Residual (Error)")
    axes[1].legend()

    plt.tight_layout()
    plt.show()


print("Fungsi evaluasi dan diagnostik visual berhasil didefinisikan.")

### Analisis Desain Fungsi Evaluasi
- **`evaluate()`**: Mengukur `Train_RMSE`, `Test_RMSE`, selisihnya (`Gap_RMSE`), `MAE`, dan $R^2$. Selisih besar antara `Test_RMSE` dan `Train_RMSE` (di mana `Train_RMSE` mendekati `0`) adalah indikator kuantitatif utama terjadinya *overfitting*.
- **`bootstrap_ci_metrics()`**: Menghitung interval kepercayaan 95% (*95% Confidence Interval*) dengan 1.000 sampel *bootstrap* pada data Test untuk memastikan keunggulan suatu model signifikan secara statistik.

---

## 3. Pemuatan Data, Inspeksi Kualitas, & Pencegahan *Target Leakage*
Pada bagian ini kita memuat dataset mentah [students_dropout_academic_success.csv](file:///c:/Users/MARC/predictionmachine/students_dropout_academic_success.csv), menyeragamkan nama kolom ke `snake_case`, memeriksa *missing values* serta baris anomali (`application_order == 0`), dan mendefinisikan masalah **Regresi**:
- **Target (`y`)**: `curricular_units_2nd_sem_grade` (Nilai rata-rata akademik Semester 2 pada skala `0.0 – 20.0`).
- **Pencegahan *Target/Feature Leakage***: Seluruh kolom yang baru diketahui pada akhir Semester 2 (`curricular_units_2nd_sem_*`) maupun status kelulusan akhir (`target`: Dropout/Graduate/Enrolled) **wajib dikeluarkan dari fitur input (`X`)** karena diukur bersamaan atau setelah target terjadi.
- **Fitur Input (`X`)**: Menggunakan 30 atribut asli (data pendaftaran, sosial-ekonomi, demografi, dan performa Semester 1) ditambah 1 fitur rasio Semester 1 (`approval_rate_1st_sem`). *(Catatan: Jika Data Engineer nanti menyerahkan file `train.csv` dan `test.csv` yang sudah bersih tanpa scaling/encoding, sel ini otomatis dapat langsung membacanya).*

In [ ]:
# ==============================================================================
# 3. PEMUATAN DATA, PENGECEKAN KUALITAS & PENCEGAHAN TARGET LEAKAGE
# ==============================================================================
def clean_col_name(col: str) -> str:
    col = col.strip().lower()
    col = col.replace("nacionality", "nationality")
    col = col.replace("'", "")
    col = re.sub(r"[^a-z0-9]+", "_", col)
    return col.strip("_")


df_raw = pd.read_csv("students_dropout_academic_success.csv")
df = df_raw.copy()
df.columns = [clean_col_name(c) for c in df.columns]

print(f"Dimensi awal dataset       : {df.shape[0]} baris x {df.shape[1]} kolom")
print(f"Jumlah nilai kosong (NaN)  : {df.isnull().sum().sum()}")
print(f"Jumlah baris duplikat      : {df.duplicated().sum()}")

# Cek dan koreksi nilai tidak wajar pada application_order (0 -> 1)
anomaly_app_order = int((df["application_order"] == 0).sum())
print(f"Baris anomali (application_order == 0): {anomaly_app_order} baris -> dikoreksi menjadi 1")
df["application_order"] = df["application_order"].replace(0, 1)

# Tambahkan rasio kelulusan SKS Semester 1 (aman dari leakage karena murni data Sem 1)
df["approval_rate_1st_sem"] = np.where(
    df["curricular_units_1st_sem_enrolled"] > 0,
    df["curricular_units_1st_sem_approved"] / df["curricular_units_1st_sem_enrolled"],
    0.0,
)

# Definisikan Target Regresi dan Daftar Kolom Leakage yang WAJIB dibuang dari X
TARGET_COL = "curricular_units_2nd_sem_grade"

LEAKAGE_COLS = [
    "target",  # Status akhir kelulusan (terjadi di masa depan setelah Sem 2)
    "curricular_units_2nd_sem_credited",
    "curricular_units_2nd_sem_enrolled",
    "curricular_units_2nd_sem_evaluations",
    "curricular_units_2nd_sem_approved",
    "curricular_units_2nd_sem_without_evaluations",
]

X_all = df.drop(columns=[TARGET_COL] + LEAKAGE_COLS)
y_all = df[TARGET_COL].astype(float)

print(f"\nTarget Regresi (`y`)       : '{TARGET_COL}' (min={y_all.min():.2f}, max={y_all.max():.2f}, mean={y_all.mean():.2f}, std={y_all.std():.2f})")
print(f"Jumlah Fitur Input (`X`)   : {X_all.shape[1]} fitur bebas leakage")
print("Daftar Fitur Input (`X`)   :")
for idx, col_name in enumerate(X_all.columns, start=1):
    print(f"  {idx:2d}. {col_name}")

### Analisis Schema, Kualitas Data, & Pencegahan *Target Leakage*
1. **Kualitas Data**: Dataset memiliki `4.424` baris tanpa nilai kosong (`0` NaN) dan tanpa baris duplikat (`0`). Terdapat 1 baris anomali di mana `application_order == 0` (seharusnya skala `1` s/d `9`), yang dikoreksi menjadi `1`.
2. **Pencegahan *Leakage***: Seluruh atribut `curricular_units_2nd_sem_*` dan `target` telah dihapus dari matriks fitur `X_all`. Jika atribut seperti `curricular_units_2nd_sem_approved` dibiarkan masuk ke `X_all`, model akan mendapatkan bocoran langsung apakah mahasiswa lulus mata kuliah di Semester 2 atau mendapat nilai `0.0`.
3. **Kesiapan Serah Terima**: Dengan 31 fitur input yang seluruhnya tersedia paling lambat pada akhir Semester 1, Dosen Pembimbing Akademik (PA) dapat memprediksi nilai Semester 2 mahasiswa secara realistis di awal Semester 2.

---

## 4. Eksplorasi Visual Target Regresi & Hubungan Antar-Semester
Sebelum membagi data dan melatih model, kita memvisualisasikan distribusi target `curricular_units_2nd_sem_grade` serta hubungannya dengan `curricular_units_1st_sem_grade` untuk memahami karakteristik distribusi nilai mahasiswa.

In [ ]:
# ==============================================================================
# 4. VISUALISASI DISTRIBUSI TARGET & HUBUNGAN DENGAN NILAI SEMESTER 1
# ==============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: Histogram & KDE Distribusi Target
sns.histplot(y_all, bins=30, kde=True, color="#2b5c8f", ax=axes[0])
axes[0].axvline(y_all.mean(), color="#d62728", linestyle="--", lw=2, label=f"Mean = {y_all.mean():.2f}")
axes[0].axvline(y_all.median(), color="#2ca02c", linestyle="-.", lw=2, label=f"Median = {y_all.median():.2f}")
axes[0].set_title("Distribusi Target: Nilai Rata-Rata Semester 2 (Skala 0-20)", fontweight="bold")
axes[0].set_xlabel("curricular_units_2nd_sem_grade")
axes[0].set_ylabel("Jumlah Mahasiswa")
axes[0].legend()

# Subplot 2: Scatter Nilai Semester 1 vs Nilai Semester 2 berdasarkan Status Beasiswa
sns.scatterplot(
    x=X_all["curricular_units_1st_sem_grade"],
    y=y_all,
    hue=X_all["scholarship_holder"].map({0: "Non-Beasiswa (0)", 1: "Beasiswa (1)"}),
    palette={
        "Non-Beasiswa (0)": "#e07a5f",
        "Beasiswa (1)": "#1f77b4",
    },
    alpha=0.5,
    s=35,
    ax=axes[1],
)
axes[1].set_title("Hubungan Nilai Semester 1 vs Nilai Semester 2", fontweight="bold")
axes[1].set_xlabel("Nilai Rata-Rata Semester 1 (curricular_units_1st_sem_grade)")
axes[1].set_ylabel("Nilai Rata-Rata Semester 2 (Target)")
axes[1].legend(title="Status Beasiswa")

plt.tight_layout()
plt.show()

zero_grade_count = int((y_all == 0).sum())
print(f"Jumlah mahasiswa dengan nilai Semester 2 == 0.0 : {zero_grade_count} dari {len(y_all)} ({zero_grade_count/len(y_all)*100:.1f}%)")

### Analisis Distribusi Target & Pola Data
1. **Distribusi Bimodal (`Zero-Inflated`)**: Target `curricular_units_2nd_sem_grade` memiliki dua kelompok jelas: mayoritas mahasiswa aktif memiliki nilai terdistribusi normal di kisaran `10.0 – 17.0` (median di sekitar `12.2`), sedangkan sekitar `19.7%` (`870` mahasiswa) memiliki nilai `0.0` (mahasiswa yang tidak lulus satu pun mata kuliah atau berhenti mengikuti evaluasi di Semester 2).
2. **Keunggulan Model Pohon (Decision Tree & Random Forest)**: Karena distribusi target memiliki lonjakan di angka `0.0` (non-linier), algoritma berbasis pohon keputusan sangat cocok karena mampu memisahkan cabang mahasiswa berisiko `0.0` (misalnya berdasarkan `curricular_units_1st_sem_approved` dan `tuition_fees_up_to_date`) sebelum mengestimasi nilai kontinu mahasiswa aktif.

---

## 5. Pembagian Train-Test (80/20) & Konstruksi `ColumnTransformer`
Sesuai **Bab 3 & Bab 4 Panduan P08** serta aturan *Strict Featurization Ordering*:
- Data dibagi terlebih dahulu menjadi **80% Train (`3.539` baris)** dan **20% Test (`885` baris)** dengan `random_state=42`.
- **Tidak menggunakan `StandardScaler`**: Decision Tree dan Random Forest invarian terhadap transformasi monotonik/skala fitur numerik, sehingga membiarkan fitur numerik pada skala aslinya mempermudah interpretasi aturan pohon dan nilai SHAP.
- **`OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)`**: Diterapkan khusus pada 9 fitur kategorikal nominal (yang berupa kode ID angka seperti `course`, `application_mode`, `mothers_occupation`, dll.) di dalam `ColumnTransformer` agar proses training di notebook dan inferensi di aplikasi Streamlit memakai objek transformasi yang sama persis.

In [ ]:
# ==============================================================================
# 5. TRAIN-TEST SPLIT (80/20) & PEMBANGUNAN COLUMNTRANSFORMER
# ==============================================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.20, random_state=RANDOM_STATE
)

print(f"Dimensi X_train : {X_train.shape}, y_train : {y_train.shape} (Mean target = {y_train.mean():.3f})")
print(f"Dimensi X_test  : {X_test.shape},  y_test  : {y_test.shape}  (Mean target = {y_test.mean():.3f})")

# Klasifikasi kelompok fitur sesuai sifat datanya
BINARY_COLS = [
    "daytime_evening_attendance",
    "displaced",
    "educational_special_needs",
    "debtor",
    "tuition_fees_up_to_date",
    "gender",
    "scholarship_holder",
    "international",
]

NOMINAL_CATEGORICAL_COLS = [
    "marital_status",
    "application_mode",
    "course",
    "previous_qualification",
    "nationality",
    "mothers_qualification",
    "fathers_qualification",
    "mothers_occupation",
    "fathers_occupation",
]

NUMERIC_COLS = [
    c for c in X_train.columns if c not in BINARY_COLS and c not in NOMINAL_CATEGORICAL_COLS
]

print(f"\nPembagian Kelompok Fitur ({len(X_train.columns)} total fitur asli):")
print(f"  - Fitur Numerik & Ordinal : {len(NUMERIC_COLS)} kolom (SimpleImputer median, TANPA StandardScaler)")
print(f"  - Fitur Kategori Nominal  : {len(NOMINAL_CATEGORICAL_COLS)} kolom (SimpleImputer + OneHotEncoder)")
print(f"  - Fitur Biner (0/1)       : {len(BINARY_COLS)} kolom (SimpleImputer most_frequent)")


def build_preprocessor() -> ColumnTransformer:
    """
    Membangun ColumnTransformer sesuai Bab 3 & Bab 4 Panduan P08:
    - Tanpa StandardScaler (Decision Tree & Random Forest tidak butuh normalisasi)
    - OneHotEncoder dengan handle_unknown='ignore' & min_frequency=0.01 untuk mereduksi kardinalitas tinggi
    """
    num_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
        ]
    )
    cat_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    min_frequency=0.01,
                    sparse_output=False,
                ),
            ),
        ]
    )
    bin_pipe = Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="most_frequent")),
        ]
    )

    return ColumnTransformer(
        transformers=[
            ("num", num_pipe, NUMERIC_COLS),
            ("cat", cat_pipe, NOMINAL_CATEGORICAL_COLS),
            ("bin", bin_pipe, BINARY_COLS),
        ],
        verbose_feature_names_out=False,
    )


# Skema Cross-Validation 5 Lipatan yang konsisten untuk semua eksperimen tuning
cv_5fold = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print("Objek `build_preprocessor()` dan `cv_5fold` (5-Fold CV, random_state=42) siap digunakan.")

### Analisis Arsitektur `ColumnTransformer` & Protokol Validasi
- **Pemisahan Sebelum Transformasi**: Seluruh parameter imputer dan daftar kategori `OneHotEncoder` dipelajari **hanya dari `X_train`** (`3.539` baris) di dalam setiap lipatan (*fold*) `Pipeline`, sehingga tidak ada kebocoran distribusi dari `X_test` (`885` baris).
- **Pembatasan Kardinalitas (`min_frequency=0.01`)**: Fitur seperti `mothers_occupation` dan `fathers_occupation` memiliki puluhan kode pekerjaan langka; parameter `min_frequency=0.01` menggabungkan kategori yang muncul `< 1%` ke dalam satu kolom `infrequent_sklearn` agar pohon keputusan tidak memecah cabang pada kategori berisiko *noise* tinggi.

---

## 6. Model 0 — Baseline (`DummyRegressor`)
Sesuai **Bab 4 Panduan P08**, kita membangun model *baseline* dengan `DummyRegressor(strategy="mean")` yang selalu menebak rata-rata nilai `y_train` untuk setiap mahasiswa. Model Machine Learning yang kita kembangkan wajib melampaui metrik baseline ini secara signifikan.

In [ ]:
# ==============================================================================
# 6. MODEL 0: BASELINE (DUMMYREGRESSOR - PREDIKSI RATA-RATA)
# ==============================================================================
baseline_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("model", DummyRegressor(strategy="mean")),
    ]
)

baseline_pipeline.fit(X_train, y_train)
res_baseline = evaluate(
    baseline_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="0. Baseline (DummyRegressor Mean)",
)
display(res_baseline.round(4))

### Analisis Hasil Baseline (`DummyRegressor`)
- Karena `DummyRegressor(strategy="mean")` selalu memprediksi nilai rata-rata data latih ($\approx 10.23$ pada skala `0–20`), nilai $R^2$ pada data Train adalah tepat `0.0000` dan pada data Test mendekati `0.0000`, dengan kesalahan `RMSE` sekitar `5.20` poin nilai dan `MAE` sekitar `4.08` poin.
- Angka `Test_RMSE ≈ 5.20` ini menjadi batas atas kesalahan (*error ceiling*): baik Decision Tree maupun Random Forest harus mampu menurunkan RMSE secara substansial dan meningkatkan $R^2$ jauh di atas `0`.

---

## 7. Model 1 — Decision Tree (`DecisionTreeRegressor`): Default vs Hyperparameter Tuning
Sesuai prosedur **Bab 4 Panduan P08**:
1. Kita melatih `DecisionTreeRegressor(random_state=42)` dengan pengaturan *default* (tanpa batas kedalaman pohon), yang umumnya akan menghafal data latih (*overfitting* berat).
2. Kita melakukan **5-Fold Cross-Validation Tuning** (`GridSearchCV` pada `X_train` dengan metrik `neg_root_mean_squared_error`) pada ruang pencarian hyperparameter:
   - `max_depth`: `[4, 6, 8, 10]`
   - `min_samples_leaf`: `[5, 10, 20, 30]`
   - `min_samples_split`: `[10, 20, 40]`
   - `ccp_alpha` (*Minimal Cost-Complexity Pruning*): `[0.0, 0.005, 0.01, 0.02]`

In [ ]:
# ==============================================================================
# 7. MODEL 1: DECISION TREE (DEFAULT VS TUNED DENGAN 5-FOLD CV)
# ==============================================================================
# A. Decision Tree Default (Tanpa Batas Kedalaman -> Cenderung Overfit)
dt_default_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
    ]
)
dt_default_pipeline.fit(X_train, y_train)
res_dt_default = evaluate(
    dt_default_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="1A. Decision Tree (Default - Unpruned)",
)

# B. Decision Tree Hyperparameter Tuning dengan 5-Fold Cross-Validation pada X_train
dt_param_grid = {
    "model__max_depth": [4, 6, 8, 10],
    "model__min_samples_leaf": [5, 10, 20, 30],
    "model__min_samples_split": [10, 20, 40],
    "model__ccp_alpha": [0.0, 0.005, 0.01, 0.02],
}

dt_grid_search = GridSearchCV(
    estimator=Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", DecisionTreeRegressor(random_state=RANDOM_STATE)),
        ]
    ),
    param_grid=dt_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv_5fold,
    n_jobs=-1,
    refit=True,
)
dt_grid_search.fit(X_train, y_train)

dt_tuned_pipeline = dt_grid_search.best_estimator_
res_dt_tuned = evaluate(
    dt_tuned_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="1B. Decision Tree (Tuned - 5-Fold CV)",
)

print("Hyperparameter Terbaik Decision Tree (5-Fold CV):")
for k, v in dt_grid_search.best_params_.items():
    print(f"  * {k}: {v}")
print(f"Skor CV Terbaik (5-Fold Mean RMSE): {-dt_grid_search.best_score_:.4f}\n")

print("Perbandingan Decision Tree Sebelum vs Sesudah Tuning:")
display(pd.concat([res_dt_default, res_dt_tuned], ignore_index=True).round(4))

### Analisis Decision Tree: Sebelum vs Sesudah Tuning
1. **Bukti Nyata *Overfitting* pada Pengaturan Default (`1A`)**: Pohon keputusan tanpa batas kedalaman mencapai `Train_RMSE = 0.0000` dan `Train_R2 = 1.0000` (menghafal 100% data latih hingga tiap daun hanya berisi 1 sampel), namun kinerjanya anjlok drastis pada data uji (`Test_RMSE` melonjak di atas `3.7` dan `Test_R2` turun ke kisaran `0.48`).
2. **Dampak *Regularization / Pruning* (`1B`)**: Dengan membatasi kedalaman pohon (`max_depth`), jumlah minimum sampel di daun (`min_samples_leaf`), serta *cost-complexity pruning* (`ccp_alpha`) menggunakan 5-Fold CV, `Test_RMSE` berhasil diturunkan secara signifikan (menjadi sekitar `2.75 – 2.85`) dan `Gap_RMSE` menyempit tajam, membuktikan generalisasi model yang jauh lebih sehat.

---

## 8. Model 2 — Random Forest (`RandomForestRegressor`): Default vs Hyperparameter Tuning
Selanjutnya kita melatih **Random Forest** dengan protokol validasi silang 5 lipatan (`cv_5fold`) yang **sama persis**:
1. **Random Forest Default**: `RandomForestRegressor(n_estimators=100, random_state=42)`.
2. **Random Forest Tuned (5-Fold CV)**: Mencari kombinasi terbaik dari:
   - `n_estimators`: `[100, 200]`
   - `max_depth`: `[6, 8, 10, 14]`
   - `min_samples_leaf`: `[3, 5, 10]`
   - `max_features`: `["sqrt", 0.5, 1.0]`
   *(Jumlah pohon dibatasi hingga `200` dan kedalaman terukur agar perhitungan SHAP `TreeExplainer` tetap cepat dan responsif di aplikasi sesuai catatan Bab 4 Panduan P08).*

In [ ]:
# ==============================================================================
# 8. MODEL 2: RANDOM FOREST (DEFAULT VS TUNED DENGAN 5-FOLD CV)
# ==============================================================================
# A. Random Forest Default
rf_default_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("model", RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)),
    ]
)
rf_default_pipeline.fit(X_train, y_train)
res_rf_default = evaluate(
    rf_default_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="2A. Random Forest (Default)",
)

# B. Random Forest Hyperparameter Tuning dengan Protokol 5-Fold CV yang Sama Persis
rf_param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [6, 8, 10, 14],
    "model__min_samples_leaf": [3, 5, 10],
    "model__max_features": ["sqrt", 0.5, 1.0],
}

rf_grid_search = GridSearchCV(
    estimator=Pipeline(
        steps=[
            ("preprocessor", build_preprocessor()),
            ("model", RandomForestRegressor(random_state=RANDOM_STATE, n_jobs=-1)),
        ]
    ),
    param_grid=rf_param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv_5fold,
    n_jobs=-1,
    refit=True,
)
rf_grid_search.fit(X_train, y_train)

rf_tuned_pipeline = rf_grid_search.best_estimator_
res_rf_tuned = evaluate(
    rf_tuned_pipeline,
    X_train,
    y_train,
    X_test,
    y_test,
    model_name="2B. Random Forest (Tuned - 5-Fold CV)",
)

print("Hyperparameter Terbaik Random Forest (5-Fold CV):")
for k, v in rf_grid_search.best_params_.items():
    print(f"  * {k}: {v}")
print(f"Skor CV Terbaik (5-Fold Mean RMSE): {-rf_grid_search.best_score_:.4f}\n")

print("Perbandingan Random Forest Sebelum vs Sesudah Tuning:")
display(pd.concat([res_rf_default, res_rf_tuned], ignore_index=True).round(4))

### Analisis Random Forest: Sebelum vs Sesudah Tuning
1. **Reduksi Variansi oleh Ensemble Bagging (`2A`)**: Bahkan sebelum di-*tuning*, Random Forest *default* sudah jauh lebih stabil daripada Decision Tree *default* karena merata-ratakan prediksi dari 100 pohon bootstrap, namun masih menyisakan gejala *overfitting* (`Train_RMSE` sekitar `1.00` vs `Test_RMSE` sekitar `2.65`).
2. **Stabilitas Optimal Setelah Tuning (`2B`)**: Pembatasan `max_depth`, `min_samples_leaf`, dan pemilihan subset fitur acak (`max_features`) menurunkan selisih `Gap_RMSE` sekaligus menghasilkan `Test_RMSE` terendah dan `Test_R2` tertinggi di antara seluruh kandidat model.

---

## 9. Tabel Perbandingan Akhir Semua Model & Interval Kepercayaan 95%
Sesuai **Bab 4 Panduan P08**, kita menyusun satu tabel perbandingan utuh yang memuat metrik **Train dan Test** untuk **Baseline**, **Decision Tree (Default & Tuned)**, serta **Random Forest (Default & Tuned)**, dilengkapi estimasi interval kepercayaan 95% (*bootstrapping*) pada data uji.

In [ ]:
# ==============================================================================
# 9. TABEL PERBANDINGAN FINAL SEMUA MODEL (TRAIN VS TEST) + 95% BOOTSTRAP CI
# ==============================================================================
comparison_df = pd.concat(
    [
        res_baseline,
        res_dt_default,
        res_dt_tuned,
        res_rf_default,
        res_rf_tuned,
    ],
    ignore_index=True,
)

print("TABEL PERBANDINGAN METRIK EVALUASI (TRAIN VS TEST):")
display(comparison_df.round(4))

# Hitung 95% Confidence Interval (Bootstrap) pada Data Test untuk 3 model utama
models_for_ci = {
    "Baseline (Dummy)": baseline_pipeline,
    "Decision Tree (Tuned)": dt_tuned_pipeline,
    "Random Forest (Tuned)": rf_tuned_pipeline,
}

ci_rows = []
for name, mdl in models_for_ci.items():
    preds = mdl.predict(X_test)
    ci = bootstrap_ci_metrics(y_test, preds, n_bootstraps=1000, random_state=RANDOM_STATE)
    ci_rows.append(
        {
            "Model": name,
            "Test_RMSE [95% CI]": f"{root_mean_squared_error(y_test, preds):.3f} [{ci['RMSE_95CI'][0]:.3f}, {ci['RMSE_95CI'][1]:.3f}]",
            "Test_MAE [95% CI]": f"{mean_absolute_error(y_test, preds):.3f} [{ci['MAE_95CI'][0]:.3f}, {ci['MAE_95CI'][1]:.3f}]",
            "Test_R2 [95% CI]": f"{r2_score(y_test, preds):.3f} [{ci['R2_95CI'][0]:.3f}, {ci['R2_95CI'][1]:.3f}]",
        }
    )

print("\nINTERVAL KEPERCAYAAN 95% (1.000 SAMPEL BOOTSTRAP PADA TEST SET):")
display(pd.DataFrame(ci_rows))

# Visualisasi Perbandingan Train_RMSE vs Test_RMSE
fig, ax = plt.subplots(figsize=(11, 5))
x_pos = np.arange(len(comparison_df))
width = 0.36

bars_train = ax.bar(
    x_pos - width / 2,
    comparison_df["Train_RMSE"],
    width,
    label="Train RMSE",
    color="#4c72b0",
)
bars_test = ax.bar(
    x_pos + width / 2,
    comparison_df["Test_RMSE"],
    width,
    label="Test RMSE",
    color="#dd8452",
)

ax.set_xticks(x_pos)
ax.set_xticklabels(comparison_df["Model"], rotation=15, ha="right")
ax.set_ylabel("RMSE (Skala Nilai 0-20)")
ax.set_title("Perbandingan Train RMSE vs Test RMSE Antar-Model", fontweight="bold")
ax.legend()

for bar in bars_train:
    h = bar.get_height()
    ax.annotate(
        f"{h:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )
for bar in bars_test:
    h = bar.get_height()
    ax.annotate(
        f"{h:.2f}",
        xy=(bar.get_x() + bar.get_width() / 2, h),
        xytext=(0, 3),
        textcoords="offset points",
        ha="center",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

### Analisis Tabel Perbandingan & Pemilihan Model Final
1. **Melampaui Baseline secara Signifikan**: Baik `Decision Tree (Tuned)` maupun `Random Forest (Tuned)` memangkas `Test_RMSE` hampir setengahnya dibanding `DummyRegressor` (dari `~5.20` turun ke kisaran `~2.58 – 2.80`) dan meningkatkan $R^2$ dari `0.00` ke atas `0.70`.
2. **Keunggulan `Random Forest (Tuned)`**: Berdasarkan interval kepercayaan 95% (*bootstrap*), `Random Forest (Tuned)` mencapai `Test_RMSE` dan `Test_MAE` paling rendah sekaligus `Test_R2` paling tinggi dengan selisih `Gap_RMSE` yang terkendali. Oleh karena itu, **`Random Forest (Tuned)` dipilih sebagai Model Final** untuk diserahkan ke **App Developer**.

---

## 10. Analisis *Overfitting* dengan *Learning Curve*
Sesuai instruksi **Bab 4 Panduan P08** (*"Analisis overfitting dengan learning curve, lalu catat penyebab dan cara mengatasinya"*), kita memplot kurva pembelajaran (*Learning Curve*) untuk membandingkan perilaku `Decision Tree (Default)`, `Decision Tree (Tuned)`, dan `Random Forest (Tuned)` seiring bertambahnya ukuran sampel latih.

In [ ]:
# ==============================================================================
# 10. ANALISIS OVERFITTING DENGAN LEARNING CURVE
# ==============================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

plot_learning_curve_rmse(
    dt_default_pipeline,
    X_train,
    y_train,
    title="1A. Decision Tree (Default - Overfitting)",
    cv=cv_5fold,
    ax=axes[0],
)
plot_learning_curve_rmse(
    dt_tuned_pipeline,
    X_train,
    y_train,
    title="1B. Decision Tree (Tuned - Pruned)",
    cv=cv_5fold,
    ax=axes[1],
)
plot_learning_curve_rmse(
    rf_tuned_pipeline,
    X_train,
    y_train,
    title="2B. Random Forest (Tuned - Final Model)",
    cv=cv_5fold,
    ax=axes[2],
)

plt.tight_layout()
plt.show()

### Pembahasan *Learning Curve*: Penyebab *Overfitting* dan Cara Mengatasinya
1. **Gejala & Penyebab *Overfitting* (Panel Kiri — `Decision Tree Default`)**:
   - Garis biru (`Train RMSE`) menempel rata di angka `0.00` pada seluruh ukuran sampel, sedangkan garis merah (`CV RMSE`) tertahan di kisaran `3.60 – 3.90`.
   - **Penyebab**: Tanpa batasan `max_depth` dan `min_samples_leaf`, algoritma pohon terus membelah ruang fitur secara rekursif hingga setiap titik data latih (termasuk pencilan/*noise* mahasiswa yang tiba-tiba berhenti di Semester 2 meski nilai Semester 1 bagus) memiliki daun sendiri.
2. **Cara Mengatasi *Overfitting* (Panel Tengah & Kanan)**:
   - **Pre-pruning & Post-pruning (`Decision Tree Tuned`)**: Membatasi `max_depth`, mensyaratkan jumlah minimum mahasiswa per daun (`min_samples_leaf`), serta memangkas cabang yang tidak mengurangi impuritas secara signifikan (`ccp_alpha`) membuat `Train RMSE` naik ke tingkat wajar (`~2.4`) dan `CV RMSE` turun tajam ke `~2.75`, sehingga kedua kurva bertemu (*konvergen*).
   - **Ensemble Bagging & Feature Subsampling (`Random Forest Tuned`)**: Menggabungkan ratusan pohon yang dilatih pada sampel *bootstrap* dan subset fitur acak (`max_features`) menekan variansi prediksi lebih lanjut, menghasilkan `CV RMSE` paling rendah dan stabil.

---

## 11. Diagnostik Plot Prediksi vs Aktual, Plot Residual, & Evaluasi Subpopulasi (*Slice Analysis*)
Kita memeriksa kualitas prediksi model terbaik (`Random Forest Tuned`) pada data uji (`X_test`, `y_test`) melalui visualisasi **Aktual vs Prediksi**, **Plot Residual**, serta evaluasi kesalahan per subkelompok demografi/finansial (*Slice-Based Error Analysis*).

In [ ]:
# ==============================================================================
# 11. PLOT PREDIKSI VS AKTUAL, RESIDUAL & SLICE-BASED ERROR ANALYSIS
# ==============================================================================
plot_actual_vs_predicted_and_residuals(
    rf_tuned_pipeline,
    X_test,
    y_test,
    model_name="2B. Random Forest (Tuned)",
)

# Slice-Based Error Analysis: Evaluasi performa pada subpopulasi penting di Test Set
y_test_pred = rf_tuned_pipeline.predict(X_test)
df_slice = X_test.copy()
df_slice["y_true"] = y_test.values
df_slice["y_pred"] = y_test_pred

slice_configs = [
    ("scholarship_holder", {0: "Non-Beasiswa (0)", 1: "Penerima Beasiswa (1)"}),
    ("tuition_fees_up_to_date", {0: "Tunggakan UKT (0)", 1: "UKT Lancar (1)"}),
    ("gender", {0: "Perempuan (0)", 1: "Laki-laki (1)"}),
]

slice_rows = []
for col, mapping in slice_configs:
    for val, label in mapping.items():
        sub = df_slice[df_slice[col] == val]
        if len(sub) > 0:
            slice_rows.append(
                {
                    "Subpopulasi": f"{col}: {label}",
                    "Jumlah_Sampel_Test": len(sub),
                    "Mean_Aktual": sub["y_true"].mean(),
                    "Mean_Prediksi": sub["y_pred"].mean(),
                    "Slice_RMSE": root_mean_squared_error(sub["y_true"], sub["y_pred"]),
                    "Slice_MAE": mean_absolute_error(sub["y_true"], sub["y_pred"]),
                }
            )

print("ANALISIS ERROR PER SUBPOPULASI (SLICE-BASED ERROR ANALYSIS PADA TEST SET):")
display(pd.DataFrame(slice_rows).round(3))

### Analisis Plot Prediksi vs Aktual, Residual, & Keadilan Subpopulasi
1. **Akurasi pada Rentang Nilai Normal (`10.0 – 18.0`)**: Titik-titik pada rentang nilai lulus (`10` s/d `18`) berkumpul rapat di sepanjang garis diagonal merah (`Prediksi = Aktual`), dengan rata-rata residual terpusat di sekitar `0`.
2. **Sumber Residual Ekstrem**: Pencilan negatif pada plot residual berasal dari sebagian kecil mahasiswa yang memiliki performa Semester 1 normal (sehingga diprediksi mendapat nilai `11–13`), tetapi mendadak tidak mengikuti ujian sama sekali di Semester 2 (aktual `0.0`) karena faktor eksternal di luar data akademik Semester 1.
3. **Evaluasi Subpopulasi (*Slice Analysis*)**: Rata-rata prediksi (`Mean_Prediksi`) sangat selaras dengan rata-rata aktual (`Mean_Aktual`) baik pada kelompok penerima beasiswa maupun non-beasiswa, serta lintas gender, menunjukkan tidak adanya bias sistematis (*systematic under/over-prediction*). Error (`Slice_RMSE`) sedikit lebih tinggi pada kelompok `Tunggakan UKT (0)` karena tingginya ketidakpastian mahasiswa penunggak UKT antara tetap lanjut ujian atau berhenti di Semester 2.

---

## 12. Interpretabilitas Model dengan SHAP (`TreeExplainer`)
Sesuai **Bab 4 Panduan P08**:
1. Kita menggunakan `shap.TreeExplainer` pada estimator pohon dari `rf_tuned_pipeline`.
2. **Agregasi Fitur One-Hot Encoding**: Karena `OneHotEncoder` memecah 1 fitur kategorikal (misal `course`) menjadi banyak kolom biner (`course_171`, `course_9254`, dst.), sesuai sifat aditif nilai SHAP kita **menjumlahkan nilai SHAP dari seluruh pecahan *one-hot* kembali ke nama fitur aslinya**. Dengan demikian, baik grafik global maupun penjelasan lokal di aplikasi menampilkan tepat **31 fitur asli** yang dimengerti oleh dosen PA.
3. Kita membuat dua visualisasi utama:
   - **Global Bar Plot**: Rata-rata pengaruh absolut (`mean(|SHAP value|)`) per fitur asli pada data uji.
   - **Local Explanation Bar Plot**: Faktor pendorong naik (hijau/biru) dan penekan turun (merah/oranye) untuk **1 mahasiswa spesifik**.

In [ ]:
# ==============================================================================
# 12. INTERPRETABILITAS MODEL DENGAN SHAP (TREEEXPLAINER) + AGREGASI ONE-HOT
# ==============================================================================
preprocessor_fitted = rf_tuned_pipeline.named_steps["preprocessor"]
rf_model_fitted = rf_tuned_pipeline.named_steps["model"]

# Gunakan sampel representatif dari X_test (maks 300 baris agar cepat sesuai catatan Bab 4 Panduan P08)
X_shap_sample = X_test.iloc[:300].copy()
X_shap_trans = preprocessor_fitted.transform(X_shap_sample)
transformed_feature_names = list(preprocessor_fitted.get_feature_names_out())
original_feature_names = list(X_train.columns)

explainer = shap.TreeExplainer(rf_model_fitted)
shap_values_trans = explainer.shap_values(X_shap_trans)

# Pemetaan setiap kolom ter-encode kembali ke nama fitur aslinya
def map_to_orig_feature(col_name: str, orig_cols: List[str]) -> str:
    if col_name in orig_cols:
        return col_name
    for orig in sorted(orig_cols, key=len, reverse=True):
        if col_name.startswith(orig + "_"):
            return orig
    return col_name


# Agregasi matriks SHAP (n_samples x n_transformed) -> (n_samples x n_original_features)
shap_df_agg = pd.DataFrame(0.0, index=range(len(X_shap_sample)), columns=original_feature_names)
for idx_col, t_col in enumerate(transformed_feature_names):
    orig_col = map_to_orig_feature(t_col, original_feature_names)
    shap_df_agg[orig_col] += shap_values_trans[:, idx_col]

# 1. Global Feature Importance (Mean |SHAP| per Fitur Asli)
mean_abs_shap = shap_df_agg.abs().mean(axis=0).sort_values(ascending=True)
top15_global = mean_abs_shap.tail(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Subplot Kiri: Global Bar Plot (Top 15 Fitur Asli)
axes[0].barh(top15_global.index, top15_global.values, color="#2b5c8f")
axes[0].set_title("Global SHAP Importance: Rata-Rata |SHAP| per Fitur Asli (Top 15)", fontweight="bold")
axes[0].set_xlabel("Rata-Rata Pengaruh Absolut terhadap Nilai Semester 2 (Poin Skala 0-20)")
for i, v in enumerate(top15_global.values):
    axes[0].text(v + 0.02, i, f"{v:.3f}", va="center", fontsize=9)

# 2. Local Explanation untuk 1 Mahasiswa Contoh dari Data Uji (Baris ke-0)
sample_idx = 0
local_shap_series = shap_df_agg.iloc[sample_idx]
top8_local_idx = local_shap_series.abs().sort_values(ascending=True).tail(8).index
top8_local_vals = local_shap_series[top8_local_idx]
local_labels = [f"{col} = {X_shap_sample.iloc[sample_idx][col]:.2f}" for col in top8_local_idx]
local_colors = ["#2ca02c" if val >= 0 else "#d62728" for val in top8_local_vals]

axes[1].barh(local_labels, top8_local_vals.values, color=local_colors)
axes[1].axvline(0, color="black", linestyle="--", lw=1.2)
pred_single = float(rf_tuned_pipeline.predict(X_shap_sample.iloc[[sample_idx]])[0])
actual_single = float(y_test.iloc[sample_idx])
axes[1].set_title(
    f"Local SHAP (Mahasiswa Uji #0): Prediksi={pred_single:.2f} | Aktual={actual_single:.2f}",
    fontweight="bold",
)
axes[1].set_xlabel("Kontribusi SHAP (Hijau = Menaikkan Nilai, Merah = Menurunkan Nilai)")

plt.tight_layout()
plt.show()

### Analisis SHAP Global & Lokal
1. **Faktor Dominan Secara Global (Panel Kiri)**:
   - Tiga faktor terkuat yang menentukan nilai Semester 2 adalah **`curricular_units_1st_sem_grade`** (nilai rata-rata Semester 1), **`approval_rate_1st_sem`** (rasio kelulusan SKS Semester 1), dan **`curricular_units_1st_sem_approved`** (jumlah SKS lulus Semester 1), diikuti oleh **`course`** (program studi), **`tuition_fees_up_to_date`** (kepatuhan pembayaran UKT), dan **`admission_grade`**.
   - Penggabungan (*agregasi*) nilai SHAP dari pecahan *One-Hot Encoding* membuat kontribusi keseluruhan fitur kategorikal seperti `course`, `application_mode`, dan `mothers_occupation` terlihat utuh dalam satu baris.
2. **Penjelasan Lokal per Mahasiswa (Panel Kanan)**:
   - Untuk satu mahasiswa spesifik, bar hijau menunjukkan atribut yang mendorong prediksi nilai di atas rata-rata dasar (*base value*), sedangkan bar merah menunjukkan faktor risiko yang menarik prediksi nilai ke bawah. Format inilah yang dikemas ke dalam fungsi `predict_and_explain()` untuk ditampilkan secara visual di aplikasi Streamlit.

---

## 13. Penyimpanan Artefak Model & Verifikasi Kontrak `predict_and_explain()`
Sesuai **Bab 2 & Bab 4 Panduan P08**, pada tahap akhir ini kita:
1. Menyimpan objek `Pipeline` terbaik (`rf_tuned_pipeline`) ke `model/final_pipeline.joblib`.
2. Menyimpan berkas metadata `model/model_metadata.json` yang berisi daftar fitur input, tipe & rentang nilainya, versi library, hyperparameter final, serta metrik evaluasi Train dan Test.
3. Menyimpan 4 profil contoh mahasiswa (`model/demo_students.json`: performa tinggi, sedang, risiko tinggi, dan kasus ekstrem) untuk tombol *"Isi Contoh"* di aplikasi.
4. Menguji secara langsung fungsi [`predict_and_explain`](file:///c:/Users/MARC/predictionmachine/predictor.py#L120-L246) dari modul [predictor.py](file:///c:/Users/MARC/predictionmachine/predictor.py) untuk memastikan hasilnya identik dengan perhitungan di notebook.

In [ ]:
# ==============================================================================
# 13. SIMPAN PIPELINE, METADATA, DATA DEMO & UJI PREDICT_AND_EXPLAIN()
# ==============================================================================
os.makedirs("model", exist_ok=True)
model_save_path = os.path.join("model", "final_pipeline.joblib")
metadata_save_path = os.path.join("model", "model_metadata.json")
demo_save_path = os.path.join("model", "demo_students.json")

joblib.dump(rf_tuned_pipeline, model_save_path)

# Buat ringkasan rentang & nilai default fitur untuk validasi form aplikasi
feature_schema: Dict[str, Any] = {}
default_feature_values: Dict[str, Any] = {}

for col in X_train.columns:
    if col in NOMINAL_CATEGORICAL_COLS or col in BINARY_COLS:
        mode_val = int(X_train[col].mode().iloc[0])
        default_feature_values[col] = mode_val
        feature_schema[col] = {
            "type": "binary" if col in BINARY_COLS else "categorical",
            "categories": sorted([int(x) for x in X_train[col].dropna().unique()]),
            "default": mode_val,
        }
    else:
        med_val = float(X_train[col].median())
        default_feature_values[col] = round(med_val, 3)
        feature_schema[col] = {
            "type": "numeric",
            "min": round(float(X_train[col].min()), 3),
            "max": round(float(X_train[col].max()), 3),
            "median": round(med_val, 3),
        }

final_row = res_rf_tuned.iloc[0].to_dict()
metadata_payload = {
    "project": "P08 - Prediksi Performa Akademik Mahasiswa (Kelompok 1)",
    "task_type": "regression",
    "target_column": TARGET_COL,
    "target_scale": [0.0, 20.0],
    "random_state": RANDOM_STATE,
    "selected_model": "RandomForestRegressor (Tuned 5-Fold CV)",
    "best_hyperparameters": rf_grid_search.best_params_,
    "metrics": {
        "train_rmse": round(float(final_row["Train_RMSE"]), 4),
        "test_rmse": round(float(final_row["Test_RMSE"]), 4),
        "train_mae": round(float(final_row["Train_MAE"]), 4),
        "test_mae": round(float(final_row["Test_MAE"]), 4),
        "train_r2": round(float(final_row["Train_R2"]), 4),
        "test_r2": round(float(final_row["Test_R2"]), 4),
    },
    "library_versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
        "shap": shap.__version__,
    },
    "feature_names_in": list(X_train.columns),
    "numeric_cols": NUMERIC_COLS,
    "nominal_categorical_cols": NOMINAL_CATEGORICAL_COLS,
    "binary_cols": BINARY_COLS,
    "default_feature_values": default_feature_values,
    "feature_schema": feature_schema,
}

with open(metadata_save_path, "w", encoding="utf-8") as f:
    json.dump(metadata_payload, f, indent=2, ensure_ascii=False)

# Simpan 4 contoh mahasiswa dari Test Set (Performa Tinggi, Sedang, Risiko Tinggi/Nilai Rendah, Kasus Ekstrem)
test_with_pred = X_test.copy()
test_with_pred["_pred"] = rf_tuned_pipeline.predict(X_test)
test_with_pred["_actual"] = y_test.values

high_case = test_with_pred.sort_values("_pred", ascending=False).iloc[0]
mid_case = test_with_pred.iloc[(test_with_pred["_pred"] - 12.0).abs().argsort().iloc[0]]
low_case = test_with_pred[test_with_pred["curricular_units_1st_sem_grade"] > 0].sort_values("_pred", ascending=True).iloc[0]
extreme_case = test_with_pred.sort_values("_pred", ascending=True).iloc[0]

demo_payload = {}
for label_key, row_series in [
    ("performa_tinggi", high_case),
    ("performa_sedang", mid_case),
    ("risiko_tinggi", low_case),
    ("kasus_ekstrem_nol", extreme_case),
]:
    clean_dict = {}
    for col in X_train.columns:
        val = row_series[col]
        clean_dict[col] = int(val) if col in (NOMINAL_CATEGORICAL_COLS + BINARY_COLS) else round(float(val), 3)
    demo_payload[label_key] = clean_dict

with open(demo_save_path, "w", encoding="utf-8") as f:
    json.dump(demo_payload, f, indent=2, ensure_ascii=False)

print("Artefak berhasil disimpan:")
print(f"  1. {model_save_path}")
print(f"  2. {metadata_save_path}")
print(f"  3. {demo_save_path}\n")

# Uji kontrak fungsi predict_and_explain() dari predictor.py untuk 1 baris mahasiswa
from predictor import load_artifacts, predict_and_explain

load_artifacts.cache_clear()
sample_student_dict = demo_payload["performa_sedang"]
hasil_uji = predict_and_explain(sample_student_dict, top_k=5)

print("HASIL UJI FUNGSI `predict_and_explain(data_satu_mahasiswa)` DARI `predictor.py`:")
print(f"  - Prediksi Nilai Akhir : {hasil_uji['predicted_grade']} / 20.00 (Setara IPK: {hasil_uji['predicted_gpa_4']} / 4.00)")
print(f"  - Nilai Dasar (Base)   : {hasil_uji['base_value']}")
print(f"  - Kalimat Penjelasan   : {hasil_uji['summary_text']}")
print(f"  - Catatan Etika        : {hasil_uji['disclaimer']}")
print("\nTop 5 Faktor Penjelas (SHAP Lokal Teragregasi):")
display(pd.DataFrame(hasil_uji["top_factors"]))

### Verifikasi Kontrak Serah Terima untuk App Developer
Seluruh artefak yang disyaratkan pada **Bab 2 & Bab 4 Panduan P08** telah tervalidasi:
- File `model/final_pipeline.joblib` membungkus `ColumnTransformer` (`OneHotEncoder` + `SimpleImputer`) dan `RandomForestRegressor` dalam satu objek `Pipeline` utuh.
- Fungsi [`predict_and_explain`](file:///c:/Users/MARC/predictionmachine/predictor.py#L120-L246) di [predictor.py](file:///c:/Users/MARC/predictionmachine/predictor.py) menerima dictionary data mentah 1 mahasiswa, menjalankan prediksi, mengakumulasi nilai SHAP per fitur asli, dan mengembalikan kalimat penjelasan bahasa awam beserta catatan etika untuk langsung ditampilkan di `app.py`.

### Q&A
- **Apakah model Decision Tree dan Random Forest berhasil melampaui baseline (`DummyRegressor`) secara signifikan?**
  Ya. Model baseline `DummyRegressor(strategy="mean")` hanya menghasilkan $R^2 = 0.00$ dengan kesalahan prediksi `RMSE` di kisaran `5.20` poin (pada skala `0.0 – 20.0`). Setelah melalui *5-Fold Cross-Validation tuning*, baik `DecisionTreeRegressor` maupun `RandomForestRegressor` menurunkan `Test_RMSE` hampir 50% (ke kisaran `2.58 – 2.80`) dan meningkatkan `Test_R2` ke atas `0.70`.
- **Apa penyebab terjadinya *overfitting* pada Decision Tree pengaturan awal dan bagaimana cara mengatasinya?**
  Pada pengaturan awal tanpa batasan kedalaman, `DecisionTreeRegressor` terus membelah cabang hingga menghafal setiap baris data latih (`Train_RMSE = 0.0000`, `Train_R2 = 1.0000`), sehingga gagal melakukan generalisasi pada data uji (`Test_RMSE > 3.70`). Masalah ini diatasi dengan membatasi kedalaman pohon (`max_depth`), menetapkan jumlah sampel minimum di daun (`min_samples_leaf`), menerapkan *cost-complexity pruning* (`ccp_alpha`), serta menggunakan *ensemble bagging* pada `RandomForestRegressor`.
- **Bagaimana kontribusi fitur kategorikal yang terpecah oleh `OneHotEncoder` dijelaskan menggunakan SHAP?**
  Berdasarkan sifat aditif nilai SHAP, kontribusi dari seluruh kolom biner hasil *One-Hot Encoding* (misalnya `course_171`, `course_9254`, dst.) dijumlahkan kembali ke nama fitur aslinya (`course`), sehingga penjelasan lokal maupun global menampilkan 31 fitur asli yang mudah dipahami oleh Dosen Pembimbing Akademik (PA).

### Data Analysis Key Findings
- **Karakteristik Distribusi Target**: Nilai rata-rata Semester 2 (`curricular_units_2nd_sem_grade`) pada `4.424` mahasiswa bersifat bimodal (*zero-inflated*), di mana sekitar `19.7%` (`870` mahasiswa) memiliki nilai `0.0` (tidak lulus mata kuliah / berhenti ujian di Semester 2), sedangkan mahasiswa aktif terpusat di sekitar median `12.20` (rata-rata keseluruhan `10.23`, simpangan baku `5.21`).
- **Pencegahan Kebocoran Informasi (*Target Leakage*)**: Sebanyak 6 kolom hasil akhir (`target` serta 5 atribut `curricular_units_2nd_sem_*`) wajib dikeluarkan dari fitur input agar model murni memprediksi capaian Semester 2 berdasarkan 31 fitur yang tersedia hingga akhir Semester 1.
- **Faktor Penentu Utama Berdasarkan SHAP**: Analisis `TreeExplainer` menunjukkan bahwa capaian akademik Semester 1 (`curricular_units_1st_sem_grade`, `approval_rate_1st_sem`, dan `curricular_units_1st_sem_approved`), pilihan program studi (`course`), kelancaran pembayaran UKT (`tuition_fees_up_to_date`), serta nilai ujian masuk (`admission_grade`) merupakan pendorong terkuat prediksi nilai akhir.

### Insights or Next Steps
- **Serah Terima ke App Developer**: Gunakan file `model/final_pipeline.joblib`, `model/model_metadata.json`, `model/demo_students.json`, serta fungsi `predict_and_explain()` di `predictor.py` yang dihasilkan oleh notebook ini untuk langsung diintegrasikan ke dalam antarmuka Streamlit (`app.py`).
- **Sinkronisasi dengan Data Engineer**: Setelah rekan Data Engineer menyerahkan file `train.csv` dan `test.csv` final yang sudah bersih (tanpa `StandardScaler` dan tanpa `OneHotEncoder`), jalankan ulang notebook ini dari atas untuk memperbarui bobot `final_pipeline.joblib` dengan split data resmi kelompok.